# 🐶 OKX-Dog 欧易量化资深专家开源小模型 Google Colab 一键极速微调训练

欢迎使用 **OKX-Dog** 专属量化模型训练笔记本！
本 Notebook 使用 **Unsloth** 深度显存优化框架，在免费的 Google Colab T4 / A100 GPU 上即可实现 7B / 14B 大模型的极速 QLoRA 训练与 GGUF 导出。

### 📌 训练流程：
1. **环境准备与 Unsloth 安装**
2. **加载量化基座模型** (`unsloth/DeepSeek-R1-Distill-Qwen-7B` 或 `unsloth/Qwen2.5-7B-Instruct`)
3. **上传并加载 OKX-Dog 导出的数据集** (`sft_train_alpaca.jsonl`)
4. **启动 SFT 监督微调**
5. **(可选) DPO 真实盈亏偏好强化**
6. **一键导出 Ollama GGUF 并生成 Modelfile**

In [ ]:
# 1. 检查 GPU 环境并安装 Unsloth
!nvidia-smi
!pip install --no-deps "xformers<0.0.29" "trl<0.9.0" peft accelerate bitsandbytes triton
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
print("✅ Unsloth 极速微调环境安装完毕！")

In [ ]:
# 2. 加载 4-bit 优化的开源量化基座模型
import torch
from unsloth import FastLanguageModel

max_seq_length = 2048 # 可根据需要调至 4096
dtype = None # 自动检测 (Tesla T4 用 float16, A100 用 bfloat16)
load_in_4bit = True # 开启 4bit 量化节省 70% 显存

# 推荐基座: unsloth/DeepSeek-R1-Distill-Qwen-7B 或 unsloth/Qwen2.5-7B-Instruct
model_name = "unsloth/DeepSeek-R1-Distill-Qwen-7B"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

# 配置 QLoRA 适配器
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)
print(f"✅ 成功加载基座模型 {model_name} 并注入 LoRA 适配层！")

In [ ]:
# 3. 上传并准备 OKX-Dog 导出的数据集 (sft_train_alpaca.jsonl)
# 如果尚未上传，可以使用下面的 files.upload() 手动选择文件
from google.colab import files
import os

dataset_filename = "sft_train_alpaca.jsonl"
if not os.path.exists(dataset_filename):
    print("请上传本地导出的 sft_train_alpaca.jsonl 文件:")
    uploaded = files.upload()

from datasets import load_dataset
dataset = load_dataset("json", data_files=dataset_filename, split="train")

# 转换为标准 ChatML 格式
def format_prompts(examples):
    instructions = examples["instruction"]
    inputs = examples["input"]
    outputs = examples["output"]
    texts = []
    for instruction, inp, output in zip(instructions, inputs, outputs):
        text = f"<|im_start|>system\n{instruction}<|im_end|>\n<|im_start|>user\n{inp}<|im_end|>\n<|im_start|>assistant\n{output}<|im_end|>"
        texts.append(text)
    return {"text": texts}

dataset = dataset.map(format_prompts, batched=True)
print(f"✅ 数据集加载成功，共包含 {len(dataset)} 条黄金训练样本！")

In [ ]:
# 4. 启动 SFT 监督微调训练
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=60, # 依据数据集大小可改为 num_train_epochs=3
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="okx_dog_intern_outputs",
    ),
)

print("🚀 开始训练微调...")
trainer_stats = trainer.train()
print("🎉 训练顺利完成！")

In [ ]:
# 5. 快速测试微调后的量化研判能力
FastLanguageModel.for_inference(model)

test_prompt = """<|im_start|>system
你是一名精通 OKX 加密货币合约量化交易的资深量化仲裁官与首席交易员。<|im_end|>
<|im_start|>user
当前市场多周期指标特征:
{"symbol": "BTC-USDT-SWAP", "current_price": 67500.0, "funding_rate": 0.0001, "indicators": {"ema": {"ema_7": 67600, "ema_25": 67200}, "macd": {"hist": 15.2}, "rsi": {"rsi_14": 58.5}}}<|im_end|>
<|im_start|>assistant
<think>
"""

inputs = tokenizer([test_prompt], return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=512, use_cache=True)
print(tokenizer.batch_decode(outputs)[0])

In [ ]:
# 6. 一键导出为 Ollama GGUF 格式 (Q4_K_M)
print("[*] 正在将模型合并并量化导出为 GGUF 格式...")
model.save_pretrained_gguf("okx_dog_intern_gguf", tokenizer, quantization_method="q4_k_m")
print("✅ GGUF 导出完成！保存于 okx_dog_intern_gguf/")

# 自动生成 Modelfile 并打包下载
modelfile_content = """FROM ./okx-dog-intern-unsloth.Q4_K_M.gguf

PARAMETER temperature 0.2
PARAMETER top_p 0.8
PARAMETER stop "<|im_end|>"

SYSTEM """你是一名精通 OKX 加密货币合约量化交易的资深量化仲裁官。"""
"""
with open("okx_dog_intern_gguf/Modelfile", "w") as f:
    f.write(modelfile_content)

print("👉 请在 Colab 左侧文件栏中下载 okx_dog_intern_gguf 中的 .gguf 文件和 Modelfile 到本地！")